# Just For Checking Data in database

## Dependencies & Constants

### Dependencies

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

### Constants

In [ ]:
project_db  = 'data/project_db.db'

## Drop Tables

In [ ]:
# # query = """
# # 	DROP TABLE yields_oats
# # """

# # query = """
# # 	DROP TABLE yields_OSR
# # """

# query = """
# 	DROP TABLE yields_spring_barley
# """

# query = """
# 	DROP TABLE yields_total_barley
# """

# query = """
# 	DROP TABLE yields_wheat
# """

# query = """
# 	DROP TABLE yields_winter_barley
# """


conn = sqlite3.connect(project_db)
try:
	cursor = conn.cursor()
	cursor.execute(query)

	conn.commit()
except Exception as e:
	print(f"Error executing query: {e}")
finally:
	conn.close()

## Query Tables

In [ ]:
query = """
	SELECT
		sb."Year" as year,
		sb."United Kingdom" AS uk_sb_yield,
		r.ann AS uk_rainfall,
		t.ann AS uk_max_temp,
		s.ann AS uk_sunshine

	FROM yields_spring_barley sb
		JOIN Rainfall r ON sb."Year" = r.year
		JOIN Tmax t ON sb."Year" = t.year
		JOIN Sunshine s ON sb."Year" = s.year
	
	WHERE r.area = 'UK'
		AND t.area = 'UK'
		AND s.area = 'UK'
		AND r.year >= 1999 AND r.year <= 2025
"""

conn = sqlite3.connect(project_db)

df_annual_uk_sb = pd.read_sql(query, conn)

conn.close()

df_annual_uk_sb.head()

## Timeseries

In [ ]:

# def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
#     plt.figure(figsize=(16,5), dpi=dpi)
#     plt.plot(x, y, color='tab:red')
#     plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
#     plt.show()

# plot_df(df, x=df.index, y=df.value, title='Monthly anti-diabetic drug sales in Australia from 1992 to 2008.')

### Rainfall data from database

In [ ]:
qry_uk_rf = """
	SELECT
		*

	FROM
		Rainfall
	WHERE
		year > 2024
		AND 
		area = 'UK'
"""

conn = sqlite3.connect(project_db)

df_uk_rf = pd.read_sql(qry_uk_rf, conn)

conn.close()

df_uk_rf.head(10)

In [ ]:
qry_uk_rf = """
	SELECT
		year,
		jan,
		feb,
		mar,
		apr,
		may,
		jun,
		jul,
		aug,
		sep,
		oct,
		nov,
		dec

	FROM
		Rainfall
	WHERE
		year >= 1998 AND year < 2026
		AND 
		area = 'UK'
"""

conn = sqlite3.connect(project_db)

df_uk_rf = pd.read_sql(qry_uk_rf, conn)

conn.close()

df_uk_rf.head(10)

### Create aggregated date field for plotting

In [ ]:
# Un-pivot the data into long format.
df_uk_rf_long = df_uk_rf.melt(id_vars='year', var_name='month', value_name='rainfall')

# Create new field for date with month and year.
df_uk_rf_long['date'] = pd.to_datetime(df_uk_rf_long['year'].astype(str) + '-' + df_uk_rf_long['month'], format='%Y-%b')

# Sort data by the date and reset the index.
df_uk_rf_long = df_uk_rf_long.sort_values('date').reset_index(drop=True)

# Verify with sample.
df_uk_rf_long.head(14)

### Plot the data

In [ ]:
def plot_df(df, x, y, title="", xlabel='Date', ylabel='Value', dpi=100):
    plt.figure(figsize=(16,5), dpi=dpi)
    plt.plot(x, y, color='tab:red')
    plt.gca().set(title=title, xlabel=xlabel, ylabel=ylabel)
    plt.show()

plot_df(df_uk_rf_long, x=df_uk_rf_long.date, y=df_uk_rf_long.rainfall, title='UK Rainfall')

This figure shows some significant fluctuation. A close look at a few single years might show a pattern.

### Try just one year

In [ ]:
df_rf_uk_2024 = df_uk_rf_long[df_uk_rf_long['year'] == 2024]

df_rf_uk_2024.head()

#### Create the plot

In [ ]:
# List of years to add.
rf_years: list = [2021, 2022, 2023]

# loop through all years in list and create dataframes.
for year in rf_years:
	df_name = f'df_rf_uk_{str(year)}'
	df_name = df_uk_rf_long[df_uk_rf_long['year'] == int(year)].copy()

	print(df_name.head(3))

In [ ]:
# # Add month number for plotting
# https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
df_rf_uk_2021.loc[:, 'month_num'] = pd.to_datetime(df_rf_uk_2021['month'], format='%b').dt.month
df_rf_uk_2022.loc[:, 'month_num'] = pd.to_datetime(df_rf_uk_2022['month'], format='%b').dt.month
df_rf_uk_2023.loc[:, 'month_num'] = pd.to_datetime(df_rf_uk_2023['month'], format='%b').dt.month
df_rf_uk_2024.loc[:, 'month_num'] = pd.to_datetime(df_rf_uk_2024['month'], format='%b').dt.month

plt.figure(figsize=(16,5))

plt.plot(df_rf_uk_2021['month_num'], df_rf_uk_2021['rainfall'], label='2021')
plt.plot(df_rf_uk_2022['month_num'], df_rf_uk_2022['rainfall'], label='2022')
plt.plot(df_rf_uk_2023['month_num'], df_rf_uk_2023['rainfall'], label='2023')
plt.plot(df_rf_uk_2024['month_num'], df_rf_uk_2024['rainfall'], label='2024')

plt.title(f'UK Monthly Rainfall ({rf_years[0]} - {rf_years[-1]})')
plt.xlabel('Month')
plt.ylabel('Rainfall (mm)')
plt.xticks(range(1,13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend()
plt.show()

This shows potential differences across the year for seasonal differences in rainfall. Depending on the sowing, growing and harvesting seasons of a crop, variations in more specific time periods could have a greater affect on yield than yearly averages.

## Analysis of Eastern England / East Anglia

### Plan

- Rainfall data for East Anglia.
- 